In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

spark = SparkSession.builder.getOrCreate()

# Generate a date spine — adjust range to suit your project history
date_df = spark.sql("""
    SELECT sequence(
        to_date('2020-01-01'),
        to_date('2030-12-01'),
        interval 1 month
    ) as period_start
""").selectExpr("explode(period_start) as period_start")

dim_billing_periods = date_df.select(
    F.monotonically_increasing_id().alias("billing_period_id"),
    F.col("period_start"),
    F.last_day(F.col("period_start")).alias("period_end"),
    F.date_format(F.col("period_start"), "MMMM yyyy").alias("billing_period_label"),
    F.month(F.col("period_start")).alias("month_number"),
    F.year(F.col("period_start")).alias("fiscal_year"),
    F.quarter(F.col("period_start")).alias("quarter")
)

dim_billing_periods.write.mode("overwrite").saveAsTable("Silver_Lakehouse.dbo.dim_billing_periods")

StatementMeta(, 138d2c35-a0f0-4e49-9657-c3fe07fd4273, 3, Finished, Available, Finished, False)